# 第 7 章: cinema データの探索と可視化

特徴量と興行収入の関係、外れ値、線形回帰の予測誤差を確認する。

In [ ]:
%use dataframe(0.15.0), kandy(0.8.0)

In [ ]:
@file:DependsOn("../build/libs/getting-started-ml.jar")

In [ ]:
import chapter02.columnMeans
import chapter02.countMissing
import chapter02.fillMissing
import chapter02.splitTrainTest
import chapter07.FEATURES
import chapter07.TARGET
import chapter07.fitLinearRegression
import chapter07.loadCinema
import chapter07.prepareCinema
import chapter07.r2Score
import chapter07.removeOutliers
import java.io.File

// Notebook は notebooks/ で実行されるので、学習データの既定の場所を 1 つ上にずらす
val cinemaCsv = File(dataset.dataDir { name -> System.getenv(name) ?: "../../data/sukkiri-ml" }, "cinema.csv")
val df = loadCinema(cinemaCsv)
countMissing(df)

## 相関を見る

In [ ]:
val numeric = dataFrameOf((FEATURES + TARGET).map { df[it].convertToDouble() })
// 欠損値を含む列は corr の対象にならないので、欠損値のある行を除いてから相関係数を求める
val corr = numeric.dropNulls().corr()
corr

In [ ]:
corr.filter { it["column"] != TARGET }.sortByDesc(TARGET).select("column", TARGET)

## 散布図で外れ値を確かめる

In [ ]:
val kept = removeOutliers(df)["cinema_id"].toList().toSet()
val marked = df.add("外れ値") { if (it["cinema_id"] in kept) "いいえ" else "はい" }
marked.plot {
    points {
        x("SNS2")
        y(TARGET)
        color("外れ値")
    }
    layout.title = "SNS2 と興行収入"
}

## 実測値と予測値、残差

In [ ]:
val split = prepareCinema(cinemaCsv, testSize = 0.2, seed = 0)
val model = fitLinearRegression(split.xTrain, split.tTrain)
val y = model.predict(split.xTest)
val results =
    dataFrameOf(
        "実測値" to split.tTest,
        "予測値" to y,
        "残差" to split.tTest.zip(y) { actual, predicted -> actual - predicted },
    )
results.plot {
    points {
        x("実測値")
        y("予測値")
    }
    layout.title = "テストデータの実測値と予測値"
}

In [ ]:
results.plot {
    points {
        x("予測値")
        y("残差")
    }
    layout.title = "予測値と残差（実測値 - 予測値）"
}

In [ ]:
results["残差"].convertToDouble().let { mapOf("最小" to it.min(), "最大" to it.max()) }

## 外れ値を除く効果を、同じテストデータで比べる

In [ ]:
fun features(frame: AnyFrame): AnyFrame = dataFrameOf(FEATURES.map { frame[it].convertToDouble() })

fun target(frame: AnyFrame): List<Double> = frame[TARGET].values().map { (it as Number).toDouble() }

fun evaluate(
    train: AnyFrame,
    test: AnyFrame,
): Map<String, Double> {
    val means = columnMeans(features(train), FEATURES)
    val fitted = fitLinearRegression(fillMissing(features(train), means), target(train))
    val predicted = fitted.predict(fillMissing(features(test), means))
    return mapOf("SNS2 の係数" to fitted.coefficients.getValue("SNS2"), "テストデータの R2" to r2Score(target(test), predicted))
}

val sameSplit = splitTrainTest(df, target(df), testSize = 0.2, seed = 0)
mapOf(
    "外れ値を残して学習" to evaluate(sameSplit.xTrain, sameSplit.xTest),
    "外れ値を除いて学習" to evaluate(removeOutliers(sameSplit.xTrain), sameSplit.xTest),
)

## 分け方によって R2 が変わる

In [ ]:
(0 until 5).associateWith { seed ->
    val s = prepareCinema(cinemaCsv, testSize = 0.2, seed = seed)
    r2Score(s.tTest, fitLinearRegression(s.xTrain, s.tTrain).predict(s.xTest))
}